In [2]:
import json
import re
import csv
from pathlib import Path

In [3]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import save_npz

In [5]:
DATASET_FOLDER = Path("/Users/megan/Downloads/PMC_Patients")

In [6]:
OUTPUT_FOLDER = DATASET_FOLDER / "tfidf_output"
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

In [7]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /Users/megan/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /Users/megan/nltk_data...
[nltk_data] Downloading package omw-1.4 to /Users/megan/nltk_data...


True

In [8]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [10]:
def preprocess_text(text):
    # Standardization: convert text to lowercase
    text = text.lower()

    # Replace punctuation and other non-alphanumeric
    # characters with spaces, preserving word boundaries
    text = re.sub(r"[^a-z0-9\s]", " ", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize the text into words
    words = text.split()

    # Remove stop words and lemmatize remaining words
    processed_words = []

    for word in words:
        if word not in stop_words:
            lemma = lemmatizer.lemmatize(word)
            processed_words.append(lemma)

    # Return the processed text as a string
    return " ".join(processed_words)

In [11]:
documents = []
document_ids = []

In [12]:
json_files = sorted(DATASET_FOLDER.rglob("*.json"))

In [13]:
print(f"Found {len(json_files)} JSON files.")

Found 8579 JSON files.


In [14]:
for file_path in json_files:

    # Skip files inside the output directory
    if OUTPUT_FOLDER in file_path.parents:
        continue

    try:
        with open(file_path, "r", encoding="utf-8") as file:
            data = json.load(file)

        # Support either one JSON object or a list of objects
        records = data if isinstance(data, list) else [data]

        for record in records:
            if not isinstance(record, dict):
                continue

            # Combine article title and patient description
            title = record.get("title", "")
            patient_text = record.get("patient", "")

            if not isinstance(title, str):
                title = ""

            if not isinstance(patient_text, str):
                patient_text = ""

            original_text = f"{title} {patient_text}".strip()

            # Skip records without usable text
            if not original_text:
                continue

            processed_text = preprocess_text(original_text)

            documents.append(processed_text)

            # Store identifiers for later reference
            document_ids.append({
                "filename": str(file_path.relative_to(DATASET_FOLDER)),
                "patient_id": record.get("patient_id", ""),
                "PMID": record.get("PMID", "")
            })

    except (json.JSONDecodeError, OSError) as error:
        print(f"Could not read {file_path}: {error}")


print(f"Successfully extracted {len(documents)} documents.")

Successfully extracted 8579 documents.


In [15]:
if not documents:
    raise ValueError(
        "No usable documents were found. "
        "Check the dataset folder and JSON structure."
    )

In [16]:
vectorizer = TfidfVectorizer(
    smooth_idf=True,
    sublinear_tf=True,
    norm="l2"
)

In [17]:
tfidf_matrix = vectorizer.fit_transform(documents)

print("TF-IDF matrix constructed successfully.")
print("Matrix dimensions:", tfidf_matrix.shape)

TF-IDF matrix constructed successfully.
Matrix dimensions: (8579, 61302)


In [18]:
save_npz(
    OUTPUT_FOLDER / "tfidf_matrix.npz",
    tfidf_matrix
)

In [19]:
vocabulary = vectorizer.get_feature_names_out()

In [20]:
with open(
    OUTPUT_FOLDER / "tfidf_vocabulary.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(vocabulary.tolist(), file, ensure_ascii=False)

In [21]:
with open(
    OUTPUT_FOLDER / "document_metadata.csv",
    "w",
    newline="",
    encoding="utf-8"
) as file:

    writer = csv.DictWriter(
        file,
        fieldnames=["filename", "patient_id", "PMID"]
    )

    writer.writeheader()
    writer.writerows(document_ids)


In [22]:
idf_values = vectorizer.idf_.tolist()

In [23]:
with open(
    OUTPUT_FOLDER / "idf_values.json",
    "w",
    encoding="utf-8"
) as file:
    json.dump(
        dict(zip(vocabulary.tolist(), idf_values)),
        file,
        ensure_ascii=False
    )

In [24]:
print(f"All output files saved to: {OUTPUT_FOLDER}")

All output files saved to: /Users/megan/Downloads/PMC_Patients/tfidf_output
